# NDR - Notebook 41a: Poison-ID Quality Probe (gated, no-submit)

## Goal
Measure whether **detection-level poison scores** can separate:
- **poison** detections (real unlearn patches transplanted)
- **clean** detections (continuous synthetic streaks)

After NB40 USE_TRIGGER=False, remaining strong hypothesis for top private scores is surgical demotion, not shared Neural Cleanse.

## Signals
| signal | source |
|---|---|
| s_diff | 1 - conf survival under cls-only empty-label FT (229/250) |
| s_sim | max cosine sim of det crop vs 20 real poison crops |
| s_geo | Mahalanobis log(w,h) vs 20 poison boxes |
| s_len_inv | shorter box more poison-like |
| blends | 229 / 250 / sim+diff |

## Gates
`
AUC_OK:     best AUC >= 0.90
SPEC_OK:    TPR@FPR_clean<=0.02 >= 0.60
SEP_OK:     median(score|poison)-median(score|clean) >= 0.25
USE_DEMOTE = all three
`
PASS -> NB41b soft-demote A/B vs 229 only. FAIL -> keep 229.


## 1. Setup


In [1]:
!pip install -q setuptools
!pip install -q git+https://github.com/facebookresearch/detectron2.git


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.4 MB/s eta 0:00:00


In [2]:
import copy, json, math, os, random, sys, traceback, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, DatasetMapper, MetadataCatalog,
    build_detection_train_loader, detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging
logging.getLogger('detectron2').setLevel(logging.ERROR)
logging.getLogger('fvcore').setLevel(logging.ERROR)

print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'ngpu', torch.cuda.device_count())

CANDIDATES = []
kaggle_input = Path('/kaggle/input')
if kaggle_input.exists():
    print('=== /kaggle/input listing ===')
    for pth in sorted(kaggle_input.iterdir()):
        print(' ', pth)
        CANDIDATES.append(pth)
        try:
            for q in pth.iterdir():
                if q.is_dir():
                    CANDIDATES.append(q)
        except Exception:
            pass
CANDIDATES.extend([
    Path('/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'),
    Path('/kaggle/input/neural-debris-removal-in-streak-detection-models'),
    Path(r'D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/data'),
])

def looks_like_root(c):
    if not c.exists() or not c.is_dir():
        return False
    w1 = c / 'poisoned_model' / 'poisoned_model.pth'
    w2 = c / 'poisoned_model.pth'
    u = c / 'unlearn_set'
    return (w1.exists() or w2.exists()) and u.exists()

ROOT = None
for c in CANDIDATES:
    try:
        if looks_like_root(c):
            ROOT = c
            break
    except Exception:
        pass
if ROOT is None and kaggle_input.exists():
    for wpath in kaggle_input.rglob('poisoned_model.pth'):
        cand = wpath.parent.parent if wpath.parent.name == 'poisoned_model' else wpath.parent
        if looks_like_root(cand):
            ROOT = cand
            break
if ROOT is None:
    raise FileNotFoundError('Cannot find competition data root. Attach competition data and rerun.')

if (ROOT / 'poisoned_model' / 'poisoned_model.pth').exists():
    POISONED_WEIGHTS = str(ROOT / 'poisoned_model' / 'poisoned_model.pth')
else:
    POISONED_WEIGHTS = str(ROOT / 'poisoned_model.pth')
UNLEARN_DIR = ROOT / 'unlearn_set'
TEST_DIR = ROOT / 'test_set' / 'test_set'
if not TEST_DIR.exists():
    TEST_DIR = ROOT / 'test_set'
if not TEST_DIR.exists():
    TEST_DIR = ROOT / 'test'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(r'D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/_probe41a')
WORK.mkdir(parents=True, exist_ok=True)
OUT_FT = WORK / 'cls_only_ft'
OUT_FT.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
CAND_THRESH = 0.05
MATCH_IOU = 0.50
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

N_BG = 24 if DEVICE == 'cuda' else 12
N_POISON_PASTE = 60 if DEVICE == 'cuda' else 30
N_CLEAN_PASTE = 60 if DEVICE == 'cuda' else 30
FT_ITERS = 20 if DEVICE == 'cuda' else 5
FT_LR = 5e-4
CROP = 64

print('ROOT=', ROOT)
print('device=', DEVICE)
print('weights exists', Path(POISONED_WEIGHTS).exists())
print('WORK=', WORK)
print('budget: N_BG=%d N_POISON=%d N_CLEAN=%d FT_ITERS=%d' % (N_BG, N_POISON_PASTE, N_CLEAN_PASTE, FT_ITERS))


python 3.12.13
torch 2.10.0+cu128 cuda True ngpu 2
=== /kaggle/input listing ===
  /kaggle/input/competitions
  /kaggle/input/datasets
ROOT= /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models
device= cuda
weights exists True
WORK= /kaggle/working
budget: N_BG=24 N_POISON=60 N_CLEAN=60 FT_ITERS=20


## 2. Loaders + poisoned model



In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    cfg.INPUT.FORMAT = 'RGB'
    if output_dir is not None:
        cfg.OUTPUT_DIR = str(output_dir)
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg

def build_predictor(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    return DefaultPredictor(build_cfg(weights, score_thresh))

def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    boxes = inst.pred_boxes.tensor.numpy().astype(np.float32)
    scores = inst.scores.numpy().astype(np.float32)
    return boxes, scores

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    aa = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    bb = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    return (inter / np.maximum(aa + bb - inter, 1e-6)).astype(np.float32)

with open(UNLEARN_DIR / 'annotations_coco.json') as f:
    coco = json.load(f)
id_to_file = {im['id']: im['file_name'] for im in coco['images']}
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}

poison_boxes_xyxy = {}
poison_wh = []
for ann in coco['annotations']:
    x, y, w, h = ann['bbox']
    poison_wh.append([w, h])
    iid = ann['image_id']
    poison_boxes_xyxy.setdefault(iid, []).append([x, y, x + w, y + h])
poison_wh = np.array(poison_wh, np.float32)

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('n unlearn', len(unlearn_files), 'n test', len(test_files), 'n anns', len(coco['annotations']))

poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('poisoned predictor ready')


n unlearn 20 n test 2000 n anns 20
poisoned predictor ready


## 3. Extract 20 real poison patches + backgrounds



In [4]:
MARGIN = 6

def extract_patch(img, xyxy, margin=MARGIN):
    x1, y1, x2, y2 = [int(round(v)) for v in xyxy]
    x1 = max(0, x1 - margin); y1 = max(0, y1 - margin)
    x2 = min(IMG_W, x2 + margin); y2 = min(IMG_H, y2 + margin)
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return img[y1:y2, x1:x2].copy(), (x1, y1, x2, y2)

poison_patches = []
missing = 0
for ann in coco['annotations']:
    iid = ann['image_id']
    fn = id_to_file[iid]
    path = UNLEARN_DIR / fn
    if not path.exists():
        cands = list(UNLEARN_DIR.rglob(Path(fn).name))
        if not cands:
            missing += 1
            continue
        path = cands[0]
    img = load_image(path)
    x, y, w, h = ann['bbox']
    xyxy = [x, y, x + w, y + h]
    out = extract_patch(img, xyxy)
    if out is None:
        missing += 1
        continue
    patch, crop_box = out
    poison_patches.append({
        'patch': patch.astype(np.float32),
        'xyxy': xyxy,
        'crop_wh': (patch.shape[1], patch.shape[0]),
        'image_id': iid,
        'stem': stem_by_id[iid],
    })

print('poison patches: %d missing=%d' % (len(poison_patches), missing))
assert len(poison_patches) >= 10, 'too few poison patches'

rng = np.random.default_rng(SEED)
n_pick = min(40, len(test_files))
idxs = rng.choice(len(test_files), size=n_pick, replace=False)
scored = []
for k in idxs:
    pth = test_files[int(k)]
    img = load_image(pth)
    with torch.no_grad():
        b, s = predict_image(poisoned_predictor, img)
    n_hi = int((s >= 0.2).sum()) if len(s) else 0
    scored.append((img, n_hi, pth.stem))
scored.sort(key=lambda t: t[1])
bgs = [im for im, _, _ in scored[:N_BG]]
print('backgrounds selected:', len(bgs), 'det_counts', [n for _, n, _ in scored[:N_BG]])


poison patches: 20 missing=0
backgrounds selected: 24 det_counts [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 4. Paste helpers + continuous streak synth



In [5]:
def paste_patch(img, patch, x0, y0, feather=2):
    out = img.astype(np.float32, copy=True)
    patch = patch.astype(np.float32, copy=False)
    ph, pw = patch.shape[:2]
    if y0 < 0 or x0 < 0 or y0 + ph > IMG_H or x0 + pw > IMG_W:
        return None
    if feather > 0:
        yy = np.linspace(0, 1, ph, dtype=np.float32)[:, None]
        xx = np.linspace(0, 1, pw, dtype=np.float32)[None, :]
        # broadcast to (ph, pw) — broadcast edge mask to (ph,pw)
        edge = np.minimum(np.minimum(yy, 1.0 - yy), np.minimum(xx, 1.0 - xx))
        alpha = np.clip(edge / max(feather / max(ph, pw), 1e-3), 0, 1).astype(np.float32)[..., None]
        region = out[y0:y0+ph, x0:x0+pw]
        if patch.ndim == 2:
            patch = np.repeat(patch[:, :, None], region.shape[2], axis=2)
        if patch.shape[2] != region.shape[2]:
            patch = np.repeat(patch[:, :, :1], region.shape[2], axis=2)
        out[y0:y0+ph, x0:x0+pw] = region * (1.0 - alpha) + patch * alpha
    else:
        if patch.ndim == 2:
            patch = np.repeat(patch[:, :, None], out.shape[2], axis=2)
        out[y0:y0+ph, x0:x0+pw] = patch
    return out

def continuous_streak_patch(pw, ph, rng, bright_scale=None, bg_level=None):
    canvas = np.zeros((ph, pw), np.float32)
    ang = rng.uniform(0, np.pi)
    L = 0.9 * min(pw, ph)
    cx, cy = pw / 2, ph / 2
    p1 = (int(cx - L/2 * np.cos(ang)), int(cy - L/2 * np.sin(ang)))
    p2 = (int(cx + L/2 * np.cos(ang)), int(cy + L/2 * np.sin(ang)))
    thick = max(1, int(round(min(pw, ph) * 0.08)))
    cv2.line(canvas, p1, p2, 1.0, thick)
    canvas = cv2.GaussianBlur(canvas, (0, 0), 0.8)
    if bright_scale is None:
        bright_scale = 40.0
    if bg_level is None:
        bg_level = 20.0
    g = bg_level + canvas / (canvas.max() + 1e-6) * bright_scale
    g = np.clip(g, 0, 255).astype(np.float32)
    return np.repeat(g[:, :, None], 3, axis=2)

def rand_free_spot(bg, pw, ph, rng, max_tries=40):
    for _ in range(max_tries):
        if IMG_W - pw <= 1 or IMG_H - ph <= 1:
            return None
        x0 = int(rng.integers(0, IMG_W - pw))
        y0 = int(rng.integers(0, IMG_H - ph))
        return x0, y0
    return None

p90 = float(np.percentile(np.concatenate([p['patch'].reshape(-1) for p in poison_patches]), 90))
p20 = float(np.percentile(np.concatenate([p['patch'].reshape(-1) for p in poison_patches]), 20))
print('poison intensity p20/p90', p20, p90)



poison intensity p20/p90 32.583656311035156 64.6926040649414


## 5. Build labeled transplant detections (poison vs clean)



In [6]:
def match_fire(boxes, scores, xyxy, conf=0.15, iou_thr=0.10):
    if len(boxes) == 0:
        return -1, 0.0
    tx1, ty1, tx2, ty2 = xyxy
    region = np.array([[tx1, ty1, tx2, ty2]], np.float32)
    ious = iou_matrix(boxes, region)[:, 0]
    cx = 0.5 * (boxes[:, 0] + boxes[:, 2])
    cy = 0.5 * (boxes[:, 1] + boxes[:, 3])
    inside = (cx >= tx1) & (cx <= tx2) & (cy >= ty1) & (cy <= ty2)
    score = ious.copy()
    score[inside] = np.maximum(score[inside], 0.15)
    order = np.argsort(-score)
    for j in order:
        if scores[j] < conf:
            continue
        if score[j] >= iou_thr or inside[j]:
            return int(j), float(scores[j])
    return -1, 0.0


def build_labeled_cases(n_poison, n_clean, seed=SEED):
    rng = np.random.default_rng(seed)
    cases = []
    tries = 0
    while len([c for c in cases if c['label'] == 1]) < n_poison and tries < n_poison * 8:
        tries += 1
        pp = poison_patches[int(rng.integers(len(poison_patches)))]
        patch = pp['patch']
        ph, pw = patch.shape[:2]
        bg = bgs[int(rng.integers(len(bgs)))]
        spot = rand_free_spot(bg, pw, ph, rng)
        if spot is None:
            continue
        x0, y0 = spot
        im2 = paste_patch(bg, patch, x0, y0)
        if im2 is None:
            continue
        xyxy = [x0 + 2, y0 + 2, x0 + pw - 2, y0 + ph - 2]
        boxes, scores = predict_image(poisoned_predictor, im2)
        j, c = match_fire(boxes, scores, xyxy)
        if j < 0:
            continue
        cases.append({
            'label': 1,
            'kind': 'poison_real',
            'image': im2,
            'xyxy': xyxy,
            'cand_box': boxes[j].astype(np.float32),
            'cand_score': float(scores[j]),
            'all_boxes': boxes,
            'all_scores': scores,
            'match_idx': j,
        })
    tries = 0
    while len([c for c in cases if c['label'] == 0]) < n_clean and tries < n_clean * 10:
        tries += 1
        wh = poison_wh[int(rng.integers(len(poison_wh)))]
        pw = int(np.clip(wh[0] * rng.uniform(0.85, 1.15), 16, 120))
        ph = int(np.clip(wh[1] * rng.uniform(0.85, 1.15), 12, 120))
        if pw % 2: pw += 1
        if ph % 2: ph += 1
        patch = continuous_streak_patch(pw, ph, rng, bright_scale=max(20.0, p90 - p20), bg_level=p20)
        bg = bgs[int(rng.integers(len(bgs)))]
        spot = rand_free_spot(bg, pw, ph, rng)
        if spot is None:
            continue
        x0, y0 = spot
        im2 = paste_patch(bg, patch, x0, y0)
        if im2 is None:
            continue
        xyxy = [x0 + 2, y0 + 2, x0 + pw - 2, y0 + ph - 2]
        boxes, scores = predict_image(poisoned_predictor, im2)
        j, c = match_fire(boxes, scores, xyxy)
        if j < 0:
            continue
        cases.append({
            'label': 0,
            'kind': 'clean_continuous',
            'image': im2,
            'xyxy': xyxy,
            'cand_box': boxes[j].astype(np.float32),
            'cand_score': float(scores[j]),
            'all_boxes': boxes,
            'all_scores': scores,
            'match_idx': j,
        })
    return cases

print('Building labeled transplant detections ...')
cases = build_labeled_cases(N_POISON_PASTE, N_CLEAN_PASTE, seed=SEED)
n_pos = sum(c['label'] == 1 for c in cases)
n_neg = sum(c['label'] == 0 for c in cases)
print('cases total=%d poison=%d clean=%d' % (len(cases), n_pos, n_neg))
print('poison cand_score mean=%.3f | clean cand_score mean=%.3f' % (
    np.mean([c['cand_score'] for c in cases if c['label']==1] or [0]),
    np.mean([c['cand_score'] for c in cases if c['label']==0] or [0]),
))
if n_pos < 15 or n_neg < 15:
    print('WARNING: few labeled detections; gates may be noisy')


Building labeled transplant detections ...
cases total=120 poison=60 clean=60
poison cand_score mean=0.573 | clean cand_score mean=0.483


## 6. Train cls-only empty-label FT (s_diff teacher)



In [7]:
UNLEARN_DATASET = 'unlearn_nb41a'

def register_unlearn():
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    dicts = []
    for im in coco['images']:
        fp = UNLEARN_DIR / im['file_name']
        if not fp.exists():
            cands = list(UNLEARN_DIR.rglob(Path(im['file_name']).name))
            fp = cands[0] if cands else fp
        dicts.append({
            'file_name': str(fp),
            'height': im['height'],
            'width': im['width'],
            'image_id': im['id'],
            'annotations': [],
        })
    DatasetCatalog.register(UNLEARN_DATASET, lambda d=dicts: d)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts

class EmptyLabelMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict['file_name'])
        dataset_dict['image'] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

class ClsOnlyTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = EmptyLabelMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(
            cfg, mapper=mapper, dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        )

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)
        for name, p in model.named_parameters():
            p.requires_grad = ('cls_score' in name or 'cls_subnet' in name)
        n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
        n_tot = sum(p.numel() for p in model.parameters())
        print('cls-only trainable %d / %d (%.1f%%)' % (n_tr, n_tot, 100*n_tr/max(n_tot,1)))
        return model

FT_WEIGHTS = OUT_FT / 'model_final.pth'
if FT_WEIGHTS.exists():
    print('reuse cached FT weights', FT_WEIGHTS)
else:
    print('Training cls-only empty-label FT: lr=%g iters=%d' % (FT_LR, FT_ITERS))
    register_unlearn()
    cfg = build_cfg(POISONED_WEIGHTS, score_thresh=0.01, output_dir=OUT_FT)
    cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.IMS_PER_BATCH = 2 if DEVICE == 'cpu' else 4
    cfg.SOLVER.BASE_LR = FT_LR
    cfg.SOLVER.MAX_ITER = FT_ITERS
    cfg.SOLVER.STEPS = []
    cfg.SOLVER.CHECKPOINT_PERIOD = 100000
    trainer = ClsOnlyTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    print('FT done', FT_WEIGHTS, 'exists', FT_WEIGHTS.exists())

if not FT_WEIGHTS.exists():
    cands = list(OUT_FT.rglob('model_final.pth'))
    if cands:
        FT_WEIGHTS = cands[0]
        print('found FT weights at', FT_WEIGHTS)
    else:
        raise FileNotFoundError('FT weights missing')

import gc
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

depoisoned_predictor = build_predictor(str(FT_WEIGHTS), score_thresh=0.01)
print('depoisoned predictor ready from', FT_WEIGHTS)


Training cls-only empty-label FT: lr=0.0005 iters=20
[07/19 14:59:18 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_f

2026-07-19 14:59:32.474219: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784473172.682783      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784473172.745251      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784473173.247883      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784473173.247913      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784473173.247915      24 computation_placer.cc:177] computation placer alr

[07/19 14:59:46 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:09 (0.5150 s / it)
[07/19 14:59:46 d2.engine.hooks]: Total training time: 0:00:25 (0:00:16 on hooks)
FT done /kaggle/working/cls_only_ft/model_final.pth exists True
[07/19 14:59:47 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/cls_only_ft/model_final.pth ...
depoisoned predictor ready from /kaggle/working/cls_only_ft/model_final.pth


## 7. Score each labeled detection



In [8]:
logwh = np.log(np.maximum(poison_wh, 1e-3))
MU = logwh.mean(0)
COV = np.cov(logwh.T) + 1e-3 * np.eye(2)
COV_INV = np.linalg.inv(COV)

def geometry_score_box(box):
    w = max(float(box[2] - box[0]), 1e-3)
    h = max(float(box[3] - box[1]), 1e-3)
    d = np.log([w, h]) - MU
    m2 = float(d @ COV_INV @ d)
    return float(np.exp(-0.5 * m2 / 2.0))

def box_length(box):
    w = float(box[2] - box[0]); h = float(box[3] - box[1])
    return float(np.hypot(w, h))

def norm_crop_gray(img, box, size=CROP):
    x1, y1, x2, y2 = [float(v) for v in box]
    cx = 0.5 * (x1 + x2); cy = 0.5 * (y1 + y2)
    side = max(x2 - x1, y2 - y1) * 1.2
    half = side / 2.0
    a = int(np.clip(cy - half, 0, IMG_H)); b = int(np.clip(cy + half, 0, IMG_H))
    c = int(np.clip(cx - half, 0, IMG_W)); d = int(np.clip(cx + half, 0, IMG_W))
    g = img[a:b, c:d, 0] if img.ndim == 3 else img[a:b, c:d]
    if g.size == 0:
        g = np.zeros((8, 8), np.float32)
    g = cv2.resize(g.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR)
    med = np.median(g); mad = np.median(np.abs(g - med)) + 1e-3
    return ((g - med) / (5 * mad)).astype(np.float32)

poison_bank = []
for pp in poison_patches:
    g = pp['patch'][:, :, 0]
    g = cv2.resize(g.astype(np.float32), (CROP, CROP), interpolation=cv2.INTER_LINEAR)
    med = np.median(g); mad = np.median(np.abs(g - med)) + 1e-3
    poison_bank.append(((g - med) / (5 * mad)).astype(np.float32))
poison_bank = np.stack(poison_bank, 0)
poison_bank_flat = poison_bank.reshape(len(poison_bank), -1)
poison_bank_flat = poison_bank_flat / (np.linalg.norm(poison_bank_flat, axis=1, keepdims=True) + 1e-6)

def sim_to_poison(img, box):
    crop = norm_crop_gray(img, box)
    v = crop.reshape(-1)
    v = v / (np.linalg.norm(v) + 1e-6)
    sims = poison_bank_flat @ v
    return float(sims.max()), float(sims.mean())

def s_diff_for_case(case):
    boxes_d, scores_d = predict_image(depoisoned_predictor, case['image'])
    cb = case['cand_box'][None, :]
    cs = case['cand_score']
    if len(boxes_d) == 0:
        return 1.0
    ious = iou_matrix(cb, boxes_d)[0]
    j = int(ious.argmax())
    if ious[j] < MATCH_IOU:
        return 1.0
    return float(1.0 - np.clip(scores_d[j] / max(cs, 1e-6), 0, 1))

rows = []
for i, case in enumerate(cases):
    sd = s_diff_for_case(case)
    smax, smean = sim_to_poison(case['image'], case['cand_box'])
    sg = geometry_score_box(case['cand_box'])
    sl = box_length(case['cand_box'])
    rows.append({
        'idx': i,
        'label': int(case['label']),
        'kind': case['kind'],
        'cand_score': case['cand_score'],
        's_diff': sd,
        's_sim': smax,
        's_sim_mean': smean,
        's_geo': sg,
        's_len': sl,
    })
    if (i + 1) % 20 == 0:
        print('scored', i + 1, '/', len(cases))

df = pd.DataFrame(rows)
med_len = float(np.median([box_length(np.array([0,0,w,h])) for w,h in poison_wh]))
df['s_len_inv'] = (med_len / np.maximum(df['s_len'].values, 1e-3)).clip(0, 3) / 3.0
df['blend_229'] = 0.90 * df['s_diff'] + 0.10 * df['s_geo']
df['blend_250'] = 0.75 * df['s_diff'] + 0.05 * df['s_geo'] + 0.20 * df['s_sim']
df['blend_simdiff'] = 0.50 * df['s_diff'] + 0.50 * df['s_sim']
df['blend_sim'] = df['s_sim']

print(df.groupby('label')[['s_diff','s_sim','s_geo','s_len','blend_229','blend_250','blend_simdiff']].median())
df.to_csv(WORK / 'probe41a_scores.csv', index=False)
print('saved', WORK / 'probe41a_scores.csv')


scored 20 / 120
scored 40 / 120
scored 60 / 120
scored 80 / 120
scored 100 / 120
scored 120 / 120
         s_diff     s_sim     s_geo      s_len  blend_229  blend_250  \
label                                                                  
0      0.753549  0.185000  0.656150  36.330646   0.752528   0.641117   
1      0.782122  0.264761  0.661866  53.285007   0.760985   0.667611   

       blend_simdiff  
label                 
0           0.473279  
1           0.505892  
saved /kaggle/working/probe41a_scores.csv


## 8. ROC / gate metrics



In [9]:
from sklearn.metrics import roc_auc_score, roc_curve

SCORE_COLS = ['s_diff', 's_sim', 's_geo', 's_len_inv', 'blend_229', 'blend_250', 'blend_simdiff']
y = df['label'].values.astype(np.int32)

def safe_auc(y_true, s):
    if len(np.unique(y_true)) < 2:
        return float('nan')
    try:
        return float(roc_auc_score(y_true, s))
    except Exception:
        return float('nan')

def tpr_at_fpr(y_true, s, fpr_max=0.02):
    if len(np.unique(y_true)) < 2:
        return 0.0, None
    fpr, tpr, thr = roc_curve(y_true, s)
    ok = np.where(fpr <= fpr_max)[0]
    if len(ok) == 0:
        return 0.0, None
    j = ok[np.argmax(tpr[ok])]
    return float(tpr[j]), float(thr[j]) if j < len(thr) else None

def sep_med(y_true, s):
    pos = s[y_true == 1]; neg = s[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    return float(np.median(pos) - np.median(neg))

metrics = []
for col in SCORE_COLS:
    s = df[col].values.astype(np.float64)
    auc = safe_auc(y, s)
    tpr02, thr02 = tpr_at_fpr(y, s, 0.02)
    tpr05, thr05 = tpr_at_fpr(y, s, 0.05)
    sep = sep_med(y, s)
    metrics.append({
        'score': col,
        'auc': auc,
        'tpr@fpr0.02': tpr02,
        'thr@fpr0.02': thr02,
        'tpr@fpr0.05': tpr05,
        'sep_med': sep,
        'med_poison': float(np.median(s[y==1])) if (y==1).any() else float('nan'),
        'med_clean': float(np.median(s[y==0])) if (y==0).any() else float('nan'),
    })

mdf = pd.DataFrame(metrics).sort_values('auc', ascending=False)
print('=== SCORE QUALITY TABLE ===')
print(mdf.to_string(index=False))
mdf.to_csv(WORK / 'probe41a_metrics.csv', index=False)

best = mdf.iloc[0]
best_name = str(best['score'])
best_auc = float(best['auc']) if best['auc'] == best['auc'] else 0.0
best_tpr = float(best['tpr@fpr0.02'])
best_sep = float(best['sep_med']) if best['sep_med'] == best['sep_med'] else 0.0

AUC_OK = best_auc >= 0.90
SPEC_OK = best_tpr >= 0.60
SEP_OK = best_sep >= 0.25
USE_DEMOTE = bool(AUC_OK and SPEC_OK and SEP_OK)

print('=' * 64)
print('NB41a POISON-ID PROBE - GATES')
print('=' * 64)
print('n_poison=%d n_clean=%d' % (int((y==1).sum()), int((y==0).sum())))
print('best_score = %s' % best_name)
print('AUC        = %.4f   (need >= 0.90)  AUC_OK=%s' % (best_auc, AUC_OK))
print('TPR@FPR2%%  = %.4f   (need >= 0.60)  SPEC_OK=%s' % (best_tpr, SPEC_OK))
print('SEP_med    = %.4f   (need >= 0.25)  SEP_OK=%s' % (best_sep, SEP_OK))
print('USE_DEMOTE = %s' % USE_DEMOTE)
print('-' * 64)
if USE_DEMOTE:
    print('PASS: detection-level poison-ID has usable separation.')
    print('Next legal step: NB41b soft-demote using best_score, A/B vs 229 only.')
    print('Do NOT hard-delete; demote conf (metric A=10).')
else:
    print('FAIL: poison-ID signals not selective enough under low clean-FPR.')
    print('Logical conclusion:')
    print('- Keep 229 as neo.')
    print('- Public demote family may already be near its measured ceiling without better labels/signals.')
    print('- Do not submit new demote/filter until a stronger measured signal appears.')
print('=' * 64)

print('\nMedian by class:')
print(df.groupby('label')[['s_diff','s_sim','s_geo','blend_229','blend_250']].agg(['median','mean']).round(3))

decision = {
    'n_poison': int((y==1).sum()),
    'n_clean': int((y==0).sum()),
    'best_score': best_name,
    'best_auc': best_auc,
    'best_tpr_fpr02': best_tpr,
    'best_sep_med': best_sep,
    'AUC_OK': AUC_OK,
    'SPEC_OK': SPEC_OK,
    'SEP_OK': SEP_OK,
    'USE_DEMOTE': USE_DEMOTE,
    'metrics': metrics,
}
with open(WORK / 'probe41a_decision.json', 'w') as f:
    json.dump(decision, f, indent=2)
print('saved', WORK / 'probe41a_decision.json')


=== SCORE QUALITY TABLE ===
        score      auc  tpr@fpr0.02  thr@fpr0.02  tpr@fpr0.05   sep_med  med_poison  med_clean
        s_sim 0.695000     0.233333     0.434348     0.266667  0.079762    0.264761   0.185000
blend_simdiff 0.629722     0.166667     0.629217     0.300000  0.032613    0.505892   0.473279
    blend_250 0.591389     0.100000     0.751230     0.183333  0.026493    0.667611   0.641117
        s_geo 0.544167     0.000000          inf     0.083333  0.005716    0.661866   0.656150
    blend_229 0.526111     0.050000     0.860330     0.100000  0.008456    0.760985   0.752528
       s_diff 0.525556     0.033333     0.939476     0.033333  0.028573    0.782122   0.753549
    s_len_inv 0.226111     0.083333     0.654088     0.100000 -0.154059    0.330290   0.484349
NB41a POISON-ID PROBE - GATES
n_poison=60 n_clean=60
best_score = s_sim
AUC        = 0.6950   (need >= 0.90)  AUC_OK=False
TPR@FPR2%  = 0.2333   (need >= 0.60)  SPEC_OK=False
SEP_med    = 0.0798   (need >= 0.25) 

## 9. Allowed conclusions



In [10]:
print('''
Interpretation rules (no guessing):
1. If USE_DEMOTE=True: poison can be identified at detection-level with low clean false-positive.
   Then 145-path remains consistent with surgical demotion; implement NB41b.
2. If AUC high but TPR@2% low: score ranks ok but threshold too soft for A=10 metric.
   Soft demote may still help small gains; hard filter will hurt.
3. If s_sim >> s_diff: local prototype beats unlearn survival (top-1 emb idea stronger than 229 s_diff).
4. If s_diff >> s_sim: stick to 229/250 survival demotion; emb filter is secondary.
5. If all fail: stop demote chase; 229 stays production.
''')
print('USE_DEMOTE=', USE_DEMOTE, 'best=', best_name)



Interpretation rules (no guessing):
1. If USE_DEMOTE=True: poison can be identified at detection-level with low clean false-positive.
   Then 145-path remains consistent with surgical demotion; implement NB41b.
2. If AUC high but TPR@2% low: score ranks ok but threshold too soft for A=10 metric.
   Soft demote may still help small gains; hard filter will hurt.
3. If s_sim >> s_diff: local prototype beats unlearn survival (top-1 emb idea stronger than 229 s_diff).
4. If s_diff >> s_sim: stick to 229/250 survival demotion; emb filter is secondary.
5. If all fail: stop demote chase; 229 stays production.

USE_DEMOTE= False best= s_sim
